# Week 4 (starter): Multi-Tool Assistant

Everything below runs with no API key. The three tools, the validator, the dispatcher, and the dispatch loop are a **worked example** in a toy domain (arithmetic and unit conversion), driven by a scripted list of tool calls. They are the reference, not your submission.

Build your assistant in a domain you choose. The **TODO (you)** comments cover Parts 1 to 4; Part 5 is the submission checklist. The example uses only the Python standard library. For live calls through the OpenAI-compatible endpoint, install the client with `pip install openai`. Setting a key alone does not enable live calls.

In [202]:
%pip install openai
%pip install python-dotenv

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [203]:
import os, ast, operator, json, math
HAS_API_KEY = bool(os.environ.get('GEMINI_API_KEY', '').strip())
print('GEMINI_API_KEY set:', HAS_API_KEY)
from openai import OpenAI# The scripted loop below runs either way. Wiring the live model call is yours (Part 1).

GEMINI_API_KEY set: True


In [204]:
print("Working directory:", os.getcwd())
print("Files:", os.listdir())

Working directory: /Users/andrew/github/cosc-650-applied-llm-systems/week-04
Files: ['.env', 'week4_tool_use_starter_Andrew.ipynb']


In [205]:
import os
from dotenv import load_dotenv

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

client = OpenAI(
    api_key=GEMINI_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

MODEL = "gemini-3.6-flash"


## Basketball Player Dataset

The assistant uses a small, hardcoded dataset of sample NBA player
statistics. The dataset is intentionally limited because the focus of
this project is function calling, schema validation, tool orchestration,
and guarded code execution rather than external data retrieval.

In [206]:
# Small mock NBA player dataset for the Basketball Stats Assistant

players = {
    "lebron_james": {
        "name": "LeBron James",
        "team": "Los Angeles Lakers",
        "position": "Forward",
        "points_per_game": 24.4,
        "rebounds_per_game": 7.8,
        "assists_per_game": 8.2,
        "games_played": 70,
        "minutes_per_game": 35.3
    },
    "stephen_curry": {
        "name": "Stephen Curry",
        "team": "Golden State Warriors",
        "position": "Guard",
        "points_per_game": 26.4,
        "rebounds_per_game": 4.5,
        "assists_per_game": 5.1,
        "games_played": 74,
        "minutes_per_game": 32.7
    },
    "nikola_jokic": {
        "name": "Nikola Jokic",
        "team": "Denver Nuggets",
        "position": "Center",
        "points_per_game": 29.6,
        "rebounds_per_game": 12.7,
        "assists_per_game": 10.2,
        "games_played": 70,
        "minutes_per_game": 36.7
    },
    "luka_doncic": {
        "name": "Luka Doncic",
        "team": "Los Angeles Lakers",
        "position": "Guard",
        "points_per_game": 28.2,
        "rebounds_per_game": 8.2,
        "assists_per_game": 7.7,
        "games_played": 50,
        "minutes_per_game": 35.1
    },
    "jayson_tatum": {
        "name": "Jayson Tatum",
        "team": "Boston Celtics",
        "position": "Forward",
        "points_per_game": 26.8,
        "rebounds_per_game": 8.7,
        "assists_per_game": 6.0,
        "games_played": 72,
        "minutes_per_game": 36.4
    },
    "giannis_antetokounmpo": {
        "name": "Giannis Antetokounmpo",
        "team": "Milwaukee Bucks",
        "position": "Forward",
        "points_per_game": 30.4,
        "rebounds_per_game": 11.9,
        "assists_per_game": 6.5,
        "games_played": 67,
        "minutes_per_game": 34.2
    }
}

In [ ]:
import ast
import signal

ALLOWED_NODES = (
    ast.Expression,
    ast.BinOp,
    ast.UnaryOp,
    ast.Constant,
    ast.Add,
    ast.Sub,
    ast.Mult,
    ast.Div,
    ast.Mod,
    ast.Pow,
    ast.USub,
    ast.UAdd,
)

# Guarded code runner: validates expressions against the arithmetic allowlist
# before allowing them to execute.
def validate_expression(expression):
    tree = ast.parse(expression, mode="eval")

    for node in ast.walk(tree):
        if not isinstance(node, ALLOWED_NODES):
            raise ValueError(
                f"Operation not allowed: {type(node).__name__}"
            )

def timeout_handler(signum, frame):
    raise TimeoutError("Calculation timed out")



# Retrieves all available basketball statistics for a specific player
# from the local player dataset.
def get_player_stats(player):
    """
    Return statistics for a player in the basketball dataset.
    """

    if player not in players:
        return {
            "success": False,
            "error": f"Player '{player}' was not found."
        }

    return {
        "success": True,
        "player": players[player]
    }

# Compares two players using a specific basketball statistic,
# such as points, rebounds, or assists per game.
def compare_players(player1, player2, stat):
    """
    Compare two players using a specific statistic.
    """

    if player1 not in players:
        return {
            "success": False,
            "error": f"Player '{player1}' was not found."
        }

    if player2 not in players:
        return {
            "success": False,
            "error": f"Player '{player2}' was not found."
        }

    if stat not in players[player1] or stat not in players[player2]:
        return {
            "success": False,
            "error": f"Stat '{stat}' was not found."
        }

    return {
        "success": True,
        "stat": stat,
        "player1": {
            "name": players[player1]["name"],
            "value": players[player1][stat]
        },
        "player2": {
            "name": players[player2]["name"],
            "value": players[player2][stat]
        }
    }

# Executes validated arithmetic with a 2-second time limit.
def stats_calculator(expression):
    validate_expression(expression)

    signal.signal(signal.SIGALRM, timeout_handler)
    signal.alarm(2)

    try:
        result = eval(expression, {"__builtins__": {}})
    finally:
        signal.alarm(0)

    return {
        "expression": expression,
        "result": result
    }


In [208]:
# Test get_player_stats
print(get_player_stats("lebron_james"))

# Test compare_players
print(
    compare_players(
        "lebron_james",
        "stephen_curry",
        "assists_per_game"
    )
)

# Test stats_calculator
print(stats_calculator("24.4 * 70"))

{'success': True, 'player': {'name': 'LeBron James', 'team': 'Los Angeles Lakers', 'position': 'Forward', 'points_per_game': 24.4, 'rebounds_per_game': 7.8, 'assists_per_game': 8.2, 'games_played': 70, 'minutes_per_game': 35.3}}
{'success': True, 'stat': 'assists_per_game', 'player1': {'name': 'LeBron James', 'value': 8.2}, 'player2': {'name': 'Stephen Curry', 'value': 5.1}}
{'expression': '24.4 * 70', 'result': 1708.0}


In [209]:
# Tool schemas for the Basketball Stats Assistant.
# These schemas tell Gemini which tools are available,
# what arguments each tool accepts, and which values are allowed.

TOOLS = [
    {
        "name": "get_player_stats",
        "description": "Retrieve basketball statistics for a specific player.",
        "parameters": {
            "type": "object",
            "properties": {
                "player": {
                    "type": "string",
                    "enum": [
                        "lebron_james",
                        "stephen_curry",
                        "nikola_jokic",
                        "luka_doncic",
                        "jayson_tatum",
                        "giannis_antetokounmpo"
                    ],
                    "description": "The player whose statistics should be retrieved."
                }
            },
            "required": ["player"],
            "additionalProperties": False
        }
    },

    {
        "name": "compare_players",
        "description": "Compare two basketball players using a specific statistic.",
        "parameters": {
            "type": "object",
            "properties": {
                "player1": {
                    "type": "string",
                    "enum": [
                        "lebron_james",
                        "stephen_curry",
                        "nikola_jokic",
                        "luka_doncic",
                        "jayson_tatum",
                        "giannis_antetokounmpo"
                    ],
                    "description": "The first player to compare."
                },

                "player2": {
                    "type": "string",
                    "enum": [
                        "lebron_james",
                        "stephen_curry",
                        "nikola_jokic",
                        "luka_doncic",
                        "jayson_tatum",
                        "giannis_antetokounmpo"
                    ],
                    "description": "The second player to compare."
                },

                "stat": {
                    "type": "string",
                    "enum": [
                        "points_per_game",
                        "rebounds_per_game",
                        "assists_per_game",
                        "games_played",
                        "minutes_per_game"
                    ],
                    "description": "The statistic used to compare the two players."
                }
            },
            "required": ["player1", "player2", "stat"],
            "additionalProperties": False
        }
    },

    {
        "name": "stats_calculator",
        "description": "Calculate a derived basketball statistic using a mathematical expression.",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "A mathematical expression containing numbers and permitted arithmetic operators."
                }
            },
            "required": ["expression"],
            "additionalProperties": False
        }
    }
]

In [210]:
IMPL = {
    "get_player_stats": get_player_stats,
    "compare_players": compare_players,
    "stats_calculator": stats_calculator
}

In [211]:
class ToolArgError(Exception): pass
# Validates the flat string/number schemas above. Extend this for your own schema types.
def validate(name, args):
    spec = next((t['parameters'] for t in TOOLS if t['name']==name), None)
    if spec is None: raise ToolArgError(f'unknown tool: {name}')
    if not isinstance(args, dict): raise ToolArgError('arguments must be a JSON object')
    for r in spec.get('required',[]):
        if r not in args: raise ToolArgError(f'missing required field: {r}')
    for k,v in args.items():
        p = spec['properties'].get(k)
        if p is None: raise ToolArgError(f'unexpected field: {k}')
        if p['type'] == 'string':
            if not isinstance(v, str): raise ToolArgError(f'{k} must be a string')
        elif p['type'] == 'number':
            if type(v) not in (int, float): raise ToolArgError(f'{k} must be a number (not a boolean)')
            if isinstance(v, float) and not math.isfinite(v): raise ToolArgError(f'{k} must be finite')
        else:
            raise ValueError(f'extend validate() to support schema type: {p["type"]}')
        if 'enum' in p and v not in p['enum']: raise ToolArgError(f'{k}={v!r} not in {p["enum"]}')
def dispatch(name, args):
    try:
        validate(name, args)
        output = IMPL[name](**args)
        json.dumps(output, allow_nan=False)  # Reject results the model cannot receive as JSON.
        return {'ok':True,'tool':name,'output':output}
    except ToolArgError as e:
        return {'ok':False,'tool':name,'error_type':'invalid_arguments','message':str(e)}
    except Exception as e:
        return {'ok':False,'tool':name,'error_type':'execution_error','message':str(e)}


In [212]:
def run_agent(query):
    messages = [{"role": "user", "content": query}]

    for _ in range(5):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=[{"type": "function", "function": t} for t in TOOLS]
        )

        message = response.choices[0].message

        # No tool call means Gemini is finished
        if not message.tool_calls:
            print("Answer:", message.content)
            return message.content

        messages.append(message)

        for call in message.tool_calls:
            name = call.function.name
            args = json.loads(call.function.arguments)

            result = dispatch(name, args)

            print(f"Tool: {name}")
            print(f"Arguments: {args}")

            if result["ok"]:
                print("Status: SUCCESS")
            else:
                print("Status: FAILED")
                print(f"Error: {result['message']}")

            print()

            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": json.dumps(result)
            })

In [213]:
_ = run_agent(
    "What are LeBron James's basketball statistics? Use the available tools."
)

Tool: get_player_stats
Arguments: {'player': 'lebron_james'}
Status: SUCCESS

Answer: Here are LeBron James's basketball statistics:

- **Team:** Los Angeles Lakers
- **Position:** Forward
- **Points Per Game:** 24.4
- **Rebounds Per Game:** 7.8
- **Assists Per Game:** 8.2
- **Games Played:** 70
- **Minutes Per Game:** 35.3


In [214]:
_ = run_agent(
    "Compare LeBron James and Stephen Curry by assists per game. Use the available tools."
)

Tool: compare_players
Arguments: {'player2': 'stephen_curry', 'stat': 'assists_per_game', 'player1': 'lebron_james'}
Status: SUCCESS

Answer: **LeBron James vs. Stephen Curry (Assists Per Game)**

* **LeBron James:** 8.2 APG
* **Stephen Curry:** 5.1 APG

LeBron James leads Stephen Curry in assists per game by 3.1 APG.


In [215]:
run_agent(
    "Get LeBron James's stats, then use the calculator tool to calculate his estimated total points by multiplying his points per game by games played. Use the tools for both steps."
)

Tool: get_player_stats
Arguments: {'player': 'lebron_james'}
Status: SUCCESS

Tool: stats_calculator
Arguments: {'expression': '24.4 * 70'}
Status: SUCCESS

Answer: LeBron James's statistics are as follows:
- **Points Per Game:** 24.4
- **Games Played:** 70
- **Rebounds Per Game:** 7.8
- **Assists Per Game:** 8.2
- **Minutes Per Game:** 35.3

Using the calculator tool to multiply his points per game (24.4) by his games played (70), his estimated total points for the season is **1,708 points** (`24.4 * 70 = 1708.0`).


"LeBron James's statistics are as follows:\n- **Points Per Game:** 24.4\n- **Games Played:** 70\n- **Rebounds Per Game:** 7.8\n- **Assists Per Game:** 8.2\n- **Minutes Per Game:** 35.3\n\nUsing the calculator tool to multiply his points per game (24.4) by his games played (70), his estimated total points for the season is **1,708 points** (`24.4 * 70 = 1708.0`)."

In [216]:
# Show the schema for the tool involved in the failure
calculator_schema = next(
    tool for tool in TOOLS
    if tool["name"] == "stats_calculator"
)

print("stats_calculator schema:")
print(json.dumps(calculator_schema["parameters"], indent=2))

run_agent(
    "Calculate LeBron James's points per game rounded to the nearest whole number using Python's round() function."
)


stats_calculator schema:
{
  "type": "object",
  "properties": {
    "expression": {
      "type": "string",
      "description": "A mathematical expression containing numbers and permitted arithmetic operators."
    }
  },
  "required": [
    "expression"
  ],
  "additionalProperties": false
}
Tool: get_player_stats
Arguments: {'player': 'lebron_james'}
Status: SUCCESS

Tool: stats_calculator
Arguments: {'expression': 'round(24.4)'}
Status: FAILED
Error: Operation not allowed: Call

Answer: LeBron James's statistics show he averages **24.4** points per game.

Using Python's `round()` function:
```python
round(24.4)  # Outputs 24
```

LeBron James's points per game rounded to the nearest whole number is **24**.


"LeBron James's statistics show he averages **24.4** points per game.\n\nUsing Python's `round()` function:\n```python\nround(24.4)  # Outputs 24\n```\n\nLeBron James's points per game rounded to the nearest whole number is **24**."

In [217]:
run_agent(
    "Get LeBron James's points per game and add 0.6 using only basic arithmetic."
)

Tool: get_player_stats
Arguments: {'player': 'lebron_james'}
Status: SUCCESS

Tool: stats_calculator
Arguments: {'expression': '24.4 + 0.6'}
Status: SUCCESS

Answer: LeBron James's points per game is **24.4**. 

Adding 0.6 to his PPG:
$$24.4 + 0.6 = 25.0$$


"LeBron James's points per game is **24.4**. \n\nAdding 0.6 to his PPG:\n$$24.4 + 0.6 = 25.0$$"

### Evaluate the Assistant

I evaluated the assistant using three queries designed to test different parts of the tool-calling workflow. The first query tested a single player-stat lookup, the second tested the player-comparison tool, and the third tested a multi-step sequence requiring the model to retrieve player statistics and then pass those results to the calculator. For each query, the tool-call log shows which tool the model selected, the arguments it provided, and whether the tool call succeeded.

1. What are LeBron James's basketball statistics? Use the available tools.

2. Compare LeBron James and Stephen Curry by assists per game. Use the available tools.

3. Get LeBron James's stats, then use the calculator tool to calculate his estimated total points by multiplying his points per game by games played. Use the tools for both steps.


### Find One Failure and Explain It


I then ran an additional query that produced a real tool failure. The query asked for LeBron James's points per game rounded to the nearest whole number using Python's `round()` function.

**Question:** Calculate LeBron James's points per game rounded to the nearest whole number using Python's `round()` function.

The model first called `get_player_stats` and successfully retrieved LeBron James's points per game as 24.4. It then called `stats_calculator` with the following argument:

`{'expression': 'round(24.4)'}`

The `stats_calculator` safely executes only approved arithmetic operations. Before running an expression, `validate_expression()` checks that the requested operation is allowed. The model attempted to use `round(24.4)`, but Python function calls such as `round()` are not included in the calculator's allowlist. The guardrail therefore blocked the operation and returned:


`Error: Operation not allowed: Call`

This was a tool execution failure because the requested expression could not be executed by the guarded code runner.

**Recovery method: Retry.**

I then ran a second query asking the model to perform an operation using only basic arithmetic:

**Question:** Get LeBron James's points per game and add 0.6 using only basic arithmetic.

The model again retrieved LeBron James's points per game as 24.4 and then called `stats_calculator` with:

`{'expression': '24.4 + 0.6'}`

This expression contains an allowlisted addition operation, so the guardrail accepted it and the tool successfully returned 25.0.

The retry demonstrates that the guarded code runner blocks unsupported Python function calls while still allowing approved arithmetic operations to execute successfully.


## Part 5: Submit
Open a pull request with your schema design write-up, a link to your notebook, and a link to an issue documenting the failure and recovery. Describe your code-runner's allowlist, time limit, and blocked operations. Rubric: schemas (20), loop including a two-step sequence (25), guarded code-runner (20), failure with recovery (20), PR hygiene (15).